# Document Loading & Text Extraction

# 01. Document Loading and Text Extraction

## Objective

Understand how different document formats expose their textual content and investigate the extraction methods required to process them.

## Document Formats

Three document formats are investigated:

1. Plain Text (TXT)
2. Markdown (MD)
3. Portable Document Format (PDF)

## Extraction Methods

| Format | Extraction Method |
|---|---|
| TXT | Python file reading |
| Markdown | Python file reading |
| PDF | PyMuPDF page-level extraction |

## Experimental Procedure

1. Define a common source text.
2. Generate controlled document fixtures.
3. Extract text from each document.
4. Preserve individual PDF page boundaries.
5. Calculate character, word, and line counts.
6. Compare the extracted content with the original source.
7. Inspect differences introduced by document formatting.

## Evaluation Metrics

- Character count
- Word count
- Line count
- Extraction success
- Exact text equality
- Normalized text equality
- PDF page count

## Limitations

The generated documents are controlled experimental fixtures.

The results demonstrate basic document extraction behavior and do not represent extraction quality across arbitrary PDFs, scanned documents, complex tables, or multi-column layouts.

In [2]:
from pathlib import Path
import json
import re

import pymupdf


# ============================================================
# 1. Project Paths
# ============================================================

PROJECT_ROOT = Path.cwd()

EXPERIMENT_DIR = PROJECT_ROOT / "data" / "experiments" / "day1_fixtures"

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. Define Controlled Source Content
# ============================================================

SOURCE_PARAGRAPHS = [
    "HADOTA Local AI Assistant",
    "HADOTA is a local AI assistant powered by Qwen through Ollama.",
    "The application uses FastAPI as its backend framework.",
    (
        "The system supports conversation management, structured outputs, "
        "and local language model inference."
    ),
    "The project is being extended with Retrieval-Augmented Generation.",
    (
        "RAG enables the assistant to retrieve relevant information "
        "from external knowledge sources before generating an answer."
    ),
]

SOURCE_TEXT = "\n\n".join(SOURCE_PARAGRAPHS)


# ============================================================
# 3. Define Document Paths
# ============================================================

TXT_PATH = EXPERIMENT_DIR / "sample.txt"
MD_PATH = EXPERIMENT_DIR / "sample.md"
PDF_PATH = EXPERIMENT_DIR / "sample.pdf"


# ============================================================
# 4. Generate TXT Document
# ============================================================

TXT_PATH.write_text(SOURCE_TEXT, encoding="utf-8")


# ============================================================
# 5. Generate Markdown Document
# ============================================================

MARKDOWN_CONTENT = "# HADOTA Local AI Assistant\n\n" + SOURCE_TEXT

MD_PATH.write_text(MARKDOWN_CONTENT, encoding="utf-8")


# ============================================================
# 6. Generate a Two-Page PDF
# ============================================================

pdf_document = pymupdf.open()

pdf_pages_content = [
    "\n\n".join(SOURCE_PARAGRAPHS[:3]),
    "\n\n".join(SOURCE_PARAGRAPHS[3:]),
]

for page_content in pdf_pages_content:
    page = pdf_document.new_page()

    remaining_space = page.insert_textbox(
        pymupdf.Rect(50, 50, 545, 780), page_content, fontsize=11
    )

    if remaining_space < 0:
        raise RuntimeError("The PDF text does not fit inside the page.")

pdf_document.save(PDF_PATH)
pdf_document.close()


# ============================================================
# 7. Extract TXT Content
# ============================================================


def extract_txt(file_path: Path) -> str:

    return file_path.read_text(encoding="utf-8")


# ============================================================
# 8. Extract Markdown Content
# ============================================================


def extract_markdown(file_path: Path) -> str:

    return file_path.read_text(encoding="utf-8")


# ============================================================
# 9. Extract PDF Content Page by Page
# ============================================================


def extract_pdf(file_path: Path) -> list[dict]:

    extracted_pages = []

    with pymupdf.open(file_path) as document:
        for page_number, page in enumerate(document.pages(), start=1):
            page_text = page.get_text("text", sort=True)

            extracted_pages.append(
                {
                    "page_number": page_number,
                    "text": page_text,
                    "character_count": len(page_text),
                    "is_empty": not page_text.strip(),
                }
            )

    return extracted_pages


# ============================================================
# 10. Load All Documents
# ============================================================

txt_text = extract_txt(TXT_PATH)

markdown_text = extract_markdown(MD_PATH)

pdf_pages = extract_pdf(PDF_PATH)

pdf_text = "\n\n".join(page["text"] for page in pdf_pages)


# ============================================================
# 11. Calculate Text Statistics
# ============================================================


def calculate_statistics(text: str) -> dict:

    return {
        "characters": len(text),
        "words": len(text.split()),
        "lines": len(text.splitlines()),
    }


results = {
    "TXT": calculate_statistics(txt_text),
    "Markdown": calculate_statistics(markdown_text),
    "PDF": calculate_statistics(pdf_text),
}


# ============================================================
# 12. Normalize Whitespace for Comparison
# ============================================================


def normalize_whitespace(text: str) -> str:

    return re.sub(r"\s+", " ", text).strip()


normalized_source = normalize_whitespace(SOURCE_TEXT)

normalized_txt = normalize_whitespace(txt_text)

normalized_markdown = normalize_whitespace(markdown_text)

normalized_pdf = normalize_whitespace(pdf_text)


# ============================================================
# 13. Compare Extracted Content
# ============================================================

comparison = {
    "TXT_exact_match": (txt_text == SOURCE_TEXT),
    "TXT_normalized_match": (normalized_txt == normalized_source),
    "Markdown_contains_source": (SOURCE_TEXT in markdown_text),
    "Markdown_has_heading": (markdown_text.startswith("# ")),
    "PDF_normalized_match": (normalized_pdf == normalized_source),
    "PDF_page_count": len(pdf_pages),
}


# ============================================================
# 14. Display Results
# ============================================================

print("=" * 65)
print("DOCUMENT LOADING AND TEXT EXTRACTION")
print("=" * 65)


print("\n1. GENERATED DOCUMENTS")

for file_path in [TXT_PATH, MD_PATH, PDF_PATH]:
    print({"name": file_path.name, "size_bytes": file_path.stat().st_size})


print("\n2. EXTRACTION STATISTICS")

print(json.dumps(results, indent=4, ensure_ascii=False))


print("\n3. CONTENT COMPARISON")

print(json.dumps(comparison, indent=4))


print("\n4. PDF PAGE INFORMATION")

print(json.dumps(pdf_pages, indent=4, ensure_ascii=False))


print("\n5. TEXT PREVIEW")

print("TXT:")
print(txt_text[:150])

print("\nMARKDOWN:")
print(markdown_text[:150])

print("\nPDF:")
print(pdf_text[:150])


print("\nEXPERIMENT COMPLETED")

DOCUMENT LOADING AND TEXT EXTRACTION

1. GENERATED DOCUMENTS
{'name': 'sample.txt', 'size_bytes': 446}
{'name': 'sample.md', 'size_bytes': 477}
{'name': 'sample.pdf', 'size_bytes': 1636}

2. EXTRACTION STATISTICS
{
    "TXT": {
        "characters": 436,
        "words": 59,
        "lines": 11
    },
    "Markdown": {
        "characters": 465,
        "words": 64,
        "lines": 13
    },
    "PDF": {
        "characters": 436,
        "words": 59,
        "lines": 13
    }
}

3. CONTENT COMPARISON
{
    "TXT_exact_match": true,
    "TXT_normalized_match": true,
    "Markdown_contains_source": true,
    "Markdown_has_heading": true,
    "PDF_normalized_match": true,
    "PDF_page_count": 2
}

4. PDF PAGE INFORMATION
[
    {
        "page_number": 1,
        "text": "HADOTA Local AI Assistant\n\nHADOTA is a local AI assistant powered by Qwen through Ollama.\n\nThe application uses FastAPI as its backend framework.",
        "character_count": 145,
        "is_empty": false
    },
  

## Result Interpretation

Extraction Statistics

| Format | Characters | Words | Lines |
|---|---:|---:|---:|
| TXT | 436 | 59 | 11 |
| Markdown | 465 | 64 | 13 |
| PDF | 436 | 59 | 13 |

The TXT extractor reproduced the original source text exactly.

The Markdown extractor preserved the original content and its additional heading syntax. The character and word counts increased because the Markdown fixture contained an additional heading.

The PDF extractor successfully recovered the source content after whitespace normalization. However, the extracted line count differed because PDF text layout introduced additional line breaks.

The PDF extraction process also preserved page-level information, producing two separate page records.

### Finding

The experiment demonstrated that TXT, Markdown, and PDF require different extraction strategies.

TXT and Markdown can be loaded through standard Python file operations, while PDF requires a dedicated extraction library such as PyMuPDF.

Page-level extraction is important for preserving document provenance and supporting future source attribution.

Whitespace normalization can help compare extracted content, but it must not be confused with general-purpose text normalization.

### Limitations

The experiment used controlled document fixtures with simple textual content.

It did not evaluate scanned PDFs, OCR, complex tables, multi-column layouts, or extraction quality across heterogeneous real-world documents.

# 02. Document Discovery

## Objective

Understand how document discovery identifies candidate files within a knowledge base.

## Key Concepts

- Directory traversal
- Recursive file discovery
- File extensions
- Supported document formats
- Deterministic ordering
- File size inspection

## Experimental Procedure

1. Inspect the existing knowledge base directory.
2. Discover files recursively.
3. Group files by extension.
4. Identify supported and unsupported formats.
5. Calculate file counts and total sizes.
6. Inspect relative file paths.

## Expected Outcome

A structured inventory of the available documents that can be used as input for the ingestion pipeline.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json

CURRENT_DIR = Path.cwd()

PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

KNOWLEDGE_BASE_DIR = PROJECT_ROOT / "knowledge base" / "raw"

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Knowledge base:", KNOWLEDGE_BASE_DIR)
print("Directory exists:", KNOWLEDGE_BASE_DIR.exists())


# Supported document formats
SUPPORTED_EXTENSIONS = {".pdf", ".md", ".txt"}


# Discover all files recursively
discovered_files = sorted(
    path for path in KNOWLEDGE_BASE_DIR.rglob("*") if path.is_file()
)


# Group files by extension
files_by_extension = defaultdict(list)

for file_path in discovered_files:
    extension = file_path.suffix.lower()

    files_by_extension[extension].append(file_path)


# Build discovery statistics
extension_counts = Counter(file_path.suffix.lower() for file_path in discovered_files)


supported_files = [
    file_path
    for file_path in discovered_files
    if file_path.suffix.lower() in SUPPORTED_EXTENSIONS
]


unsupported_files = [
    file_path
    for file_path in discovered_files
    if file_path.suffix.lower() not in SUPPORTED_EXTENSIONS
]


total_size = sum(file_path.stat().st_size for file_path in discovered_files)


# Prepare inventory
inventory = {
    "root_directory": str(KNOWLEDGE_BASE_DIR),
    "total_files": len(discovered_files),
    "supported_files": len(supported_files),
    "unsupported_files": len(unsupported_files),
    "total_size_bytes": total_size,
    "extension_counts": dict(extension_counts),
    "files": [
        {
            "name": file_path.name,
            "relative_path": str(file_path.relative_to(PROJECT_ROOT)),
            "extension": file_path.suffix.lower(),
            "size_bytes": file_path.stat().st_size,
            "supported": (file_path.suffix.lower() in SUPPORTED_EXTENSIONS),
        }
        for file_path in discovered_files
    ],
}


print("=" * 65)
print("DOCUMENT DISCOVERY EXPERIMENT")
print("=" * 65)

print(json.dumps(inventory, indent=4, ensure_ascii=False))

Current directory: d:\Projects\Local Mini RAG Document QA System\notebooks
Project root: d:\Projects\Local Mini RAG Document QA System
Knowledge base: d:\Projects\Local Mini RAG Document QA System\knowledge base\raw
Directory exists: True
DOCUMENT DISCOVERY EXPERIMENT
{
    "root_directory": "d:\\Projects\\Local Mini RAG Document QA System\\knowledge base\\raw",
    "total_files": 6,
    "supported_files": 6,
    "unsupported_files": 0,
    "total_size_bytes": 118951,
    "extension_counts": {
        ".md": 6
    },
    "files": [
        {
            "name": "README_03_Local_Runtime_Engineering.md",
            "relative_path": "knowledge base\\raw\\README_03_Local_Runtime_Engineering.md",
            "extension": ".md",
            "size_bytes": 14492,
            "supported": true
        },
        {
            "name": "README_04_prompt_construction_and_structured_output.md",
            "relative_path": "knowledge base\\raw\\README_04_prompt_construction_and_structured_output.m

## Result Interpretation

### Discovery Statistics

| Metric | Result |
|---|---:|
| Total Files | 6 |
| Supported Files | 6 |
| Unsupported Files | 0 |
| Total Size | 118,951 Bytes |
| Markdown Files | 6 |
| TXT Files | 0 |
| PDF Files | 0 |

### Findings

The document discovery process successfully identified all six files in the configured raw knowledge base directory.

All discovered files use the Markdown extension, which is included in the supported extension allowlist.

The discovery process uses recursive directory traversal and deterministic sorting to produce a reproducible file inventory.

Project-relative paths were generated successfully after correcting the project root resolution.

### Engineering Observations

The experiment confirmed that document discovery should be independent of document extraction.

Discovery identifies candidate files and collects filesystem metadata without reading or interpreting their contents.

The current corpus contains only Markdown files. Additional fixtures will be required to evaluate unsupported extensions, nested directories, empty files, inaccessible files, and file size restrictions.

### Limitations

The experiment evaluated the existing raw knowledge base only.

It did not test malformed files, filesystem permission errors, symbolic links, path traversal attempts, or large-file rejection.

The results describe the current corpus and should not be generalized to arbitrary document collections.

# 03. Document Schema Design

## Objective

Understand how structured document records represent source files and their extracted content.

## Key Concepts

- Document identity
- Source provenance
- File metadata
- Content hashing
- Extraction metadata
- Document records versus page records
- Schema validation

## Document Record

A document record represents an original source file and stores its extracted content alongside its identifying information.

## Page Record

A page record represents an individual page extracted from a document.

Page-level records are particularly important for PDF documents because they preserve page provenance for future source attribution.

## Design Requirements

The schema should:

1. Provide a stable document identifier.
2. Preserve the original source path.
3. Store the original file name and extension.
4. Store the extracted content.
5. Support content hashing for change detection.
6. Preserve file size and ingestion metadata.
7. Support optional page-level information.
8. Validate data types and required fields.

## Expected Outcome

A clear distinction between document-level and page-level data models that can be implemented using Pydantic.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from hashlib import sha256
import json

PROJECT_ROOT = Path.cwd().parent

sample_file = (
    PROJECT_ROOT / "knowledge base" / "raw" / "README_03_Local_Runtime_Engineering.md"
)

content = sample_file.read_text(encoding="utf-8")
file_bytes = sample_file.read_bytes()

document_record = {
    "document_id": sha256(
        str(sample_file.relative_to(PROJECT_ROOT)).encode("utf-8")
    ).hexdigest()[:16],
    "file_name": sample_file.name,
    "source_path": str(sample_file.relative_to(PROJECT_ROOT)),
    "file_type": sample_file.suffix.lower(),
    "content": content,
    "content_hash": sha256(file_bytes).hexdigest(),
    "file_size": sample_file.stat().st_size,
    "ingested_at": datetime.now(timezone.utc).isoformat(),
    "metadata": {"encoding": "utf-8", "source_type": "local_file"},
}

print("DOCUMENT RECORD")
print("=" * 60)

print(
    json.dumps(
        {key: value for key, value in document_record.items() if key != "content"},
        indent=4,
        ensure_ascii=False,
    )
)

print("\nCONTENT PREVIEW")
print("=" * 60)
print(document_record["content"][:500])

print("\nVALIDATION")
print("=" * 60)

print("Document ID exists:", bool(document_record["document_id"]))
print("Content exists:", bool(document_record["content"]))
print("Content hash length:", len(document_record["content_hash"]))
print("File size matches:", len(file_bytes) == document_record["file_size"])

DOCUMENT RECORD
{
    "document_id": "5824500a14b09ea1",
    "file_name": "README_03_Local_Runtime_Engineering.md",
    "source_path": "knowledge base\\raw\\README_03_Local_Runtime_Engineering.md",
    "file_type": ".md",
    "content_hash": "1aa2321b3340f18c43df43e935cf7634cfca5b9dfd71c903afc39c5d1e5e064e",
    "file_size": 14492,
    "ingested_at": "2026-11-01T21:07:05.368592+00:00",
    "metadata": {
        "encoding": "utf-8",
        "source_type": "local_file"
    }
}

CONTENT PREVIEW
# Local LLM Runtime Engineering

## Overview

Local LLM Runtime Engineering focuses on building a structured runtime integration layer for the Local AI Assistant.

The implementation separates application-level generation requests from the underlying LLM runtime. It provides validated request and response schemas, centralized configuration, an Ollama HTTP adapter, normalized streaming events, runtime error handling, and automated tests.

The goal is to establish a maintainable runtime foundation

V

## Result Interpretation

### Document Record Inspection

The experiment successfully generated a structured document record from the source Markdown file.

The record contains the document identifier, original file name, relative source path, file extension, content hash, file size, ingestion timestamp, and additional metadata.

### Validation Results

| Validation | Result |
|---|---|
| Document ID exists | Passed |
| Extracted content exists | Passed |
| SHA-256 hash length is 64 characters | Passed |
| Recorded file size matches actual file size | Passed |

### Findings

The experiment demonstrated how document identity and content integrity can be represented using separate identifiers.

The document ID was derived from the relative source path, while the content hash was calculated using SHA-256 over the original file bytes.

This distinction allows the system to identify a source document and detect changes to its underlying content independently.

The ingestion timestamp provides a record of when the document entered the ingestion process.

### Important Observation

The file size validation confirmed that the recorded byte count matches the actual source file size.

However, this check verifies size consistency only. It does not establish that the file content is correct, complete, or uncorrupted.

### Limitations

The experiment evaluated a single Markdown document.

It did not test document updates, file relocation, duplicate content, hash collisions, or multiple ingestion runs.